In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import sys

# Handling large CSV strings
import csv
csv.field_size_limit = sys.maxsize

# 1. Load Data
df = pd.read_csv('zomato.csv', engine='python', on_bad_lines='warn')

# 2. Advanced Cleaning (The "Expert" way)
df.drop_duplicates(inplace=True)

# Handle the 'rate' column: Remove '/5', handle 'NEW'/'-', and convert to float
df['rate'] = df['rate'].astype(str).replace(['NEW', '-', 'nan', 'NaN'], np.nan)
df['rate'] = df['rate'].str.replace('/5', '').str.strip()
df['rate'] = pd.to_numeric(df['rate'], errors='coerce')

# Handle the 'cost' column: Remove commas and handle types
df['approx_cost(for two people)'] = df['approx_cost(for two people)'].astype(str).str.replace(',', '')
df['approx_cost(for two people)'] = pd.to_numeric(df['approx_cost(for two people)'], errors='coerce')

# 3. Handling Missing Values (Imputation)
# Dropping rows where we have NO rating or cost (Target variables)
df = df.dropna(subset=['rate', 'approx_cost(for two people)'])

# Filling categorical nulls so we don't lose data for location/cuisine analysis
df['rest_type'] = df['rest_type'].fillna('Other')
df['cuisines'] = df['cuisines'].fillna('Other')

# 4. Feature Engineering (Proving "Actionable Insights" [cite: 4])
# Create a 'services_offered' count (Online Order + Book Table)
df['services_count'] = (df['online_order'] == 'Yes').astype(int) + (df['book_table'] == 'Yes').astype(int)

# 5. SQL Integration (To support your Resume Bullet [cite: 17])
conn = sqlite3.connect('zomato_analysis.db')
df.to_sql('restaurants', conn, if_exists='replace', index=False)

# 6. Save for Power BI
df.to_csv('zomato_clean.csv', index=False)

print("Data cleaning complete. Cleaned shape:", df.shape)